This is the old one.kind of complicated

In [27]:
# =============================================================================
# Simulated Morlet-theta-power CCA @ 100 Hz (matches your real pipeline)
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import cwt, morlet2, fftconvolve
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr

# ---------------------- global config ---------------------------------
FS = 100  # Hz
DT = 1 / FS
RNG = np.random.default_rng(7)

FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']
N_CH = len(FRONTAL_MIDLINE)

# Your paper-like Morlet settings
FREQS    = np.arange(1, 46)  # 1..45 Hz
N_CYCLES = np.logspace(np.log10(3), np.log10(12), len(FREQS))  # len=45

# Theta band definition (inclusive)
THETA_LOW = 4
THETA_HIGH = 8

# ---------------------- helpers ---------------------------------------
def zscore(x, axis=0, eps=1e-12):
    m = np.mean(x, axis=axis, keepdims=True)
    s = np.std(x, axis=axis, ddof=0, keepdims=True)
    return (x - m) / np.maximum(s, eps)

def percent_baseline(power_f_by_t, baseline_idx):
    """
    power_f_by_t: (n_freq, T) power (>=0)
    baseline_idx: boolean or integer indices for baseline time samples
    Returns power with 'percent' baseline across time, per frequency:
        100 * (x - mean_baseline) / mean_baseline
    """
    x = power_f_by_t.copy()
    b = x[:, baseline_idx].mean(axis=1, keepdims=True)  # (n_freq, 1)
    # avoid /0
    b = np.where(b == 0, 1e-12, b)
    return 100.0 * (x - b) / b

def morlet_tfr_power(x, fs, freqs, n_cycles):
    """
    x: (T,) real signal
    freqs: (F,) array of frequencies in Hz
    n_cycles: (F,) array of Morlet cycles (one per freq) - like in your paper
    Returns: power (F, T) = |x ⋆ ψ_f|^2 using morlet2 per-frequency.
    This avoids scipy.signal.cwt deprecation and supports per-freq n_cycles.
    """
    T = len(x)
    power = np.empty((len(freqs), T), dtype=float)

    # Choose wavelet length per freq ~ 10 cycles worth of support (like SciPy did)
    # M ≈ 10 * s  where s is "scale" (std dev in samples) used by morlet2
    # scale s = n_cycles * fs / (2π f)
    two_pi = 2 * np.pi
    for i, (f, nc) in enumerate(zip(freqs, n_cycles)):
        s = nc * fs / (two_pi * f)                  # scale in samples
        M = int(np.clip(np.round(10 * s), 32, 8_192))  # cap sane bounds
        # build wavelet and convolve (same)
        wv = morlet2(M, s, w=nc)                    # complex Morlet with nc cycles
        conv = fftconvolve(x, np.conj(wv[::-1]), mode="same")
        power[i, :] = (np.abs(conv) ** 2).real

    return power  # (F, T)

def morlet_theta_power_multich(raw_eeg, fs, freqs, n_cycles, baseline_s=(0.0, 2.0)):
    """
    raw_eeg: (T, C) raw AM signals per channel.
    Pipeline per channel:
      raw -> Morlet power 1..45 Hz (per-freq cycles) -> percent baseline (per freq)
      -> average over theta (4-8 Hz)
    Returns: (T, C) theta power (percent-baselined, no per- channel z-scoring).
    """
    T, C = raw_eeg.shape
    t = np.arange(T) / fs
    base_mask = (t >= baseline_s[0]) & (t <= baseline_s[1])
    theta_idx = np.where((freqs >= THETA_LOW) & (freqs <= THETA_HIGH))[0]

    out = np.empty((T, C), dtype=float)
    for c in range(C):
        p_f_t = morlet_tfr_power(raw_eeg[:, c], fs, freqs, n_cycles)  # (F, T)
        p_bl  = percent_baseline(p_f_t, base_mask)                    # (F, T)
        theta_ts = p_bl[theta_idx].mean(axis=0)                       # (T,)
        out[:, c] = theta_ts

    return out

def make_latent_driver(T_sec, fs, f_mean=6.0, f_jitter=0.6):
    """
    Latent 'cognitive' driver z(t), standardized, that sets the envelope dynamics.
    Combines a 6 Hz-ish oscillation with a slow on/off/plateau envelope.
    """
    T = int(round(T_sec * fs))
    t = np.arange(T) / fs

    f_inst = f_mean + f_jitter * np.sin(2*np.pi*0.03*t)
    phase = 2*np.pi * np.cumsum(f_inst) * (1/fs)
    carrier = np.sin(phase)

    slow_on = 0.5*np.tanh((t - 2.0)/1.2) - 0.5*np.tanh((t - (T_sec - 2.0))/1.2)
    plateau = 0.6 + 0.4*np.tanh((t - 6.0)/2.0) - 0.4*np.tanh((t - (T_sec - 6.0))/2.0)
    amp = 0.6*slow_on + 0.4*plateau

    z = np.maximum(amp * carrier, 0.0) + 0.25*amp
    z = (z - z.mean()) / z.std(ddof=0)
    return z  # (T,)

def delay_signal(x, fs, lag_ms):
    """Positive lag_ms means output lags input (shift right)."""
    s = int(np.round(lag_ms * fs / 1000.0))
    if s == 0:
        return x.copy()
    if s > 0:
        return np.r_[np.full(s, x[0]), x[:-s]]
    else:
        return np.r_[x[-s:], np.full(-s, x[-1])]

def synthesize_raw_eeg_from_envelope(env, fs, n_ch=N_CH,
                                     eeg_noise_sd=0.2, cross_talk=0.15,
                                     f_car=6.0):
    """
    Build raw channel signals: channel_c(t) = gain_c * env(t) * cos(2π f_car t + φ_c)
    + small cross-talk + noise. Returns (raw, gains).
    """
    T = len(env)
    t = np.arange(T) / fs
    gains = np.maximum(1.0 + 0.5 * RNG.normal(size=n_ch), 0.1)

    raw = np.zeros((T, n_ch))
    for c in range(n_ch):
        phi = RNG.uniform(0, 2*np.pi)
        carrier = np.cos(2*np.pi*f_car*t + phi)
        ch = gains[c] * env * carrier
        if c > 0:
            ch += cross_talk * raw[:, :c].sum(axis=1) / max(1, c)
        ch += eeg_noise_sd * RNG.normal(size=T)
        raw[:, c] = ch

    return raw, gains

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over time×channels."""
    x = x - x.mean(axis=0, keepdims=True)         # de-mean per channel
    scale = np.sqrt(np.mean(x**2))                # one global RMS
    return x / max(scale, 1e-12)                  # avoid /0


def candidate_lags_units(step_ms=10, max_ms=1000):
    """Returns integer sample shifts for ±max_ms in step_ms steps (@100 Hz)."""
    steps_ms = np.arange(-max_ms, max_ms + step_ms, step_ms)
    return (steps_ms / 10).astype(int)  # 10 ms/sample at 100 Hz

def align_by_shift_segment(X, y, shift_samples):
    """
    Align EEG (X: TxC) to pupil (y: T,) by DROPPING edges.
    Positive shift => EEG lags are dropped from the START (EEG leads PPD).
    Negative shift => drop from the START of PPD.

    Returns: (Xs, ys) with identical length; or (None, None) if nothing left.
    """
    T = min(len(X), len(y))
    if shift_samples >= 0:
        s = shift_samples
        if T <= s:
            return None, None
        Xs = X[s:]            # drop early EEG
        ys = y[:T - s]        # drop late pupil
    else:
        s = -shift_samples
        if T <= s:
            return None, None
        Xs = X[:T - s]        # drop late EEG
        ys = y[s:]            # drop early pupil
    return Xs, ys

def concat_trials(trials, shift_samples=0, trim_ms=None):
    """
    trials: list of dicts with 'X':(T,C), 'y':(T,)
    Applies SEGMENT alignment per trial (no circular roll).
    Optional extra symmetric trim (in ms) after alignment to reduce edge effects.
    """
    Xs, Ys = [], []
    k = int(round((trim_ms or 0) / 10))  # 10 ms/sample @ 100 Hz

    for tr in trials:
        X = tr['X']; y = tr['y']
        Xs_al, ys_al = align_by_shift_segment(X, y, shift_samples)
        if Xs_al is None:
            continue
        if k > 0:
            if len(ys_al) <= 2*k:    # guard: too short after trim
                continue
            Xs_al = Xs_al[k:-k]
            ys_al = ys_al[k:-k]
        Xs_al = normalise_eeg(Xs_al)
        ys_al = (ys_al - ys_al.mean()) / ys_al.std(ddof=0)
        Xs.append(Xs_al)
        Ys.append(ys_al)

    if not Xs:
        return np.empty((0, trials[0]['X'].shape[1])), np.empty((0,))
    return np.vstack(Xs), np.concatenate(Ys)


def cca_corr(X, y):
    """1D CCA correlation between X and y (single component)."""
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(X, y.reshape(-1, 1))
    u, v = cca.transform(X, y.reshape(-1, 1))
    return float(pearsonr(u[:, 0], v[:, 0])[0])

def search_best_lag(train_trials, shifts, return_curve=False, trim_ms=None):
    r_per_shift = {}
    for s in shifts:
        X, y = concat_trials(train_trials, shift_samples=s, trim_ms=trim_ms)
        if len(y) == 0:
            r_per_shift[s] = np.nan
            continue
        r_per_shift[s] = cca_corr(X, y)
    # pick best over valid shifts
    valid = {k: v for k, v in r_per_shift.items() if np.isfinite(v)}
    best_s = max(valid, key=valid.get)
    best_r = valid[best_s]
    if return_curve:
        return best_r, best_s, r_per_shift
    return best_r, best_s, None

# ---------------------- dataset simulation ----------------------------
def simulate_dataset(
    n_trials=12,
    trial_len_s=26.0,
    pupil_lag_ms=350,
    eeg_noise_sd=0.30,
    pupil_noise_sd=0.15,
    pupil_smoothing_ms=180,
    cross_talk=0.15,
    baseline_s=(0.0, 2.0),
    perfect_mode=False,  # if True, bypass wavelets: X = gains * env directly
):
    """
    Returns a list of trials; each trial is dict:
      {'X': theta_power(T,C), 'y': pupil(T,), 'z': latent(T,), 'w_true': (C,)}
    """
    trials = []
    for _ in range(n_trials):
        # 1) latent driver and its (positive) envelope for AM
        z = make_latent_driver(trial_len_s, FS)
        env = 0.8 + 0.6 * z
        # optional: energy normalize without removing positivity
        env = env / np.sqrt(np.mean(env**2))

        # 2) raw EEG per channel (or perfect-mode features)
        if perfect_mode:
            gains = np.maximum(1.0 + 0.5 * RNG.normal(size=N_CH), 0.1)
            theta_pow = np.outer(env, gains)  # already a "feature" matrix
        else:
            raw, gains = synthesize_raw_eeg_from_envelope(
                env, FS, n_ch=N_CH, eeg_noise_sd=eeg_noise_sd,
                cross_talk=cross_talk, f_car=6.0
            )
            theta_pow = morlet_theta_power_multich(
                raw, FS, freqs=FREQS, n_cycles=N_CYCLES, baseline_s=baseline_s
            )

        # 3) pupil follows the *same envelope* (to match theta power), delayed
        y = delay_signal(env, FS, pupil_lag_ms)

        # optional smoothing (moving average)
        if pupil_smoothing_ms and pupil_smoothing_ms > 0:
            w = max(1, int(round(pupil_smoothing_ms / 1000 * FS)))
            if w > 1:
                y = np.convolve(y, np.ones(w)/w, mode='same')

        # add noise and standardize
        y += pupil_noise_sd * RNG.normal(size=len(y))
        y = (y - y.mean()) / y.std(ddof=0)

        trials.append({
            'X': theta_pow, 'y': y, 'z': z, 'w_true': gains / np.linalg.norm(gains), 'env': env
        })
    return trials

# ------------------------- main demo ----------------------------------
if __name__ == "__main__":
    # ---- knobs ----
    TRUE_LAG_MS = 100
    EEG_NOISE   = 0.30
    PUP_NOISE   = 0
    PUP_SMOOTH  = 180   # ms
    CROSS_TALK  = 0.15
    BASELINE_S  = (0.0, 2.0)  # seconds
    TRIM_MS     = 300   # ms, to avoid edge effects in CCA lag search

    PERFECT_MODE = False  # set True for "pure linear" sanity check

    # 1) simulate trials
    trials = simulate_dataset(
        n_trials=12,
        trial_len_s=26.0,
        pupil_lag_ms=TRUE_LAG_MS,
        eeg_noise_sd=EEG_NOISE,
        pupil_noise_sd=PUP_NOISE,
        pupil_smoothing_ms=PUP_SMOOTH,
        cross_talk=CROSS_TALK,
        baseline_s=BASELINE_S,
        perfect_mode=PERFECT_MODE,
    )

    # 2) lag search
    shifts = candidate_lags_units(step_ms=10, max_ms=1000)  # ±1 s in 10 ms steps
    best_r, best_shift_samp, curve = search_best_lag(
        trials, shifts, return_curve=True, trim_ms=TRIM_MS  # trim avoids Morlet edges
    )
    best_shift_ms = int(best_shift_samp * 10)

    print(f"\nGround-truth lag:   {TRUE_LAG_MS} ms")
    print(f"Recovered best lag: {best_shift_ms} ms with r = {best_r:.3f}")

    # 3) fit CCA once at best lag, evaluate
    X_all, y_all = concat_trials(trials, shift_samples=best_shift_samp, trim_ms=TRIM_MS)
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(X_all, y_all.reshape(-1, 1))
    u_all, v_all = cca.transform(X_all, y_all.reshape(-1, 1))
    r_overall = pearsonr(u_all[:, 0], v_all[:, 0])[0]
    print(f"Whole-data canonical correlation at best lag: r = {r_overall:.3f}")

    # 4) plots
    # r vs lag
    xs = np.array(sorted(curve.keys())) * 10
    ys = np.array([curve[k] for k in sorted(curve.keys())])
    plt.figure(figsize=(6.2, 3.2))
    plt.plot(xs, ys, lw=2)
    plt.axvline(TRUE_LAG_MS, ls=':', label=f"true = {TRUE_LAG_MS} ms")
    plt.axvline(best_shift_ms, ls='--', label=f"best = {best_shift_ms} ms")
    plt.xlabel("Lag (ms)  [EEG → PPD]")
    plt.ylabel("Canonical r")
    plt.title("CCA correlation vs lag (simulated)")
    plt.grid(alpha=.3); plt.legend(); plt.tight_layout()

    # take the first trial for time-domain overlays
    tr = trials[0]
    X1 = tr['X']
    y1 = tr['y']
    z1 = tr['z']
    env1 = tr['env']
    w_true = tr['w_true']

    # align EEG features to pupil by applying best lag to X
    X1_shift2, pupil_ts2 = align_by_shift_segment(X1, y1, best_shift_samp)
    K = int(round(300/10))
    X1_shift = X1[K:-K]
    pupil_ts = y1[K:-K]
    
    Tmin = min(len(X1), len(y1))
    s = best_shift_samp

    env_del = zscore(delay_signal(env1, FS, TRUE_LAG_MS)) #why would i want to delay this?
    z_del   = zscore(delay_signal(z1,   FS, TRUE_LAG_MS)) #why?

    if s >= 0:
        # pupil kept first (Tmin - s) samples before the K trim
        env_seg = env_del[:Tmin - s]
        z_seg   = z_del[:Tmin - s]
    else:
        s = -s
        # pupil kept samples starting at s
        env_seg = env_del[s:]
        z_seg   = z_del[s:]

    # now apply the same K trim
    env_aligned = env_del[K:-K]
    z_aligned   = z1[K:-K]

    # true vs recovered 1D components
    eeg_true_comp = zscore(X1_shift @ w_true)  # "true" comp (features • true weights)
    w_est = cca.x_weights_[:, 0]
    w_est2 = [1]*N_CH
    eeg_rec_comp = zscore(X1_shift @ w_est)
    eeg_rec_comp2 = zscore(X1_shift @ w_est2)


    T1 = len(pupil_ts)
    t = np.arange(T1) / FS


    plt.figure(figsize=(9.4, 4.0))
    plt.plot(t, pupil_ts, label="Pupil (PPD)")
    plt.plot(t, z_aligned, label="Latent z1(t) (aligned)")
    plt.plot(t, eeg_true_comp, label="True EEG comp (X·w_true)")
    plt.plot(t, eeg_rec_comp, label="Recovered EEG comp (X·w_CCA)")
    plt.plot(t, eeg_rec_comp2, label="Recovered EEG comp2 (X·w_CCA)")
    plt.xlim(2, t[-1]-2)
    plt.xlabel("Time (s)")
    plt.ylabel("Z-scored amplitude")
    plt.title("Trial 1: pupil, latent, true EEG comp, recovered EEG comp")
    plt.grid(alpha=.3); plt.legend(ncol=2); plt.tight_layout()

    # weights bar plot
    plt.figure(figsize=(9.6, 4.1))
    idx = np.arange(N_CH)
    cos_sim = float(np.dot(w_true, w_est) /
                    (np.linalg.norm(w_true) * np.linalg.norm(w_est)))
    plt.bar(idx - 0.2, w_true / np.linalg.norm(w_true), width=0.4, label="true (normed)")
    plt.bar(idx + 0.2, w_est / np.linalg.norm(w_est),  width=0.4, label="CCA (normed)")
    plt.xticks(idx, FRONTAL_MIDLINE, rotation=45)
    plt.ylabel("Weight (normalized)")
    plt.title(f"Electrode weights — true vs recovered (cosine = {cos_sim:.3f})")
    plt.legend(); plt.tight_layout()

    plt.show()


C:\Users\cdd\AppData\Local\Temp\ipykernel_752\2353280317.py:64: DeprecationWarning: scipy.signal.morlet2 is deprecated in SciPy 1.12 and will be removed
in SciPy 1.15. We recommend using PyWavelets instead.

  wv = morlet2(M, s, w=nc)                    # complex Morlet with nc cycles



Ground-truth lag:   100 ms
Recovered best lag: -20 ms with r = 0.947
Whole-data canonical correlation at best lag: r = 0.947


In [6]:
%matplotlib qt